In [23]:
import numpy as np
mat_size=32
input_width=8
result_width=32

In [24]:
# 32x32 행렬 2개 생성 (8비트 부호 있는 정수: -128 ~ 127)
# np.random.seed(42)  # 재현성을 위한 시드 설정
max_num=2**(input_width-1)
if (input_width==8):
    d_type=np.int8
    h_type=np.uint8 # uint8로 수정

matrix_a = np.random.randint(-max_num, max_num, size=(mat_size, mat_size), dtype=d_type)
matrix_b = np.random.randint(-max_num, max_num, size=(mat_size, mat_size), dtype=d_type)
matrix_b_trans=matrix_b.T

# 8비트 2의 보수 형식으로 변환 (0x00 ~ 0xFF)
matrix_a_hex = matrix_a.astype(h_type)
matrix_b_hex = matrix_b.astype(h_type)
matrix_b_trans_hex = matrix_b_trans.astype(h_type)



In [25]:
# matrix_a에서 절반을 랜덤으로 0으로 만들어 sparse_a 생성
sparse_a = matrix_a.copy()

# 전체 인덱스 중 절반을 랜덤 선택
total_elements = mat_size * mat_size
indices = np.random.choice(total_elements, size=total_elements // 2, replace=False)

# 선택된 인덱스를 0으로 설정
rows = indices // mat_size
cols = indices % mat_size
sparse_a[rows, cols] = 0

# sparse_a 정보 출력
sparse_a_hex = sparse_a.astype(h_type)

print(f"=== Sparse Matrix A ===")
print(f"Total elements: {sparse_a.size}")
print(f"Non-zero elements: {np.count_nonzero(sparse_a)}")
print(f"Sparsity: {100 * (1 - np.count_nonzero(sparse_a) / sparse_a.size):.2f}%")

print(f"\nSparse A (top-left 4x4):")
print(sparse_a[:4, :4])

print(f"\nSparse A (top-left 4x4, Hex):")
for row in sparse_a_hex[:4, :4]:
    print(" ".join(f"{val:02X}" for val in row))

=== Sparse Matrix A ===
Total elements: 1024
Non-zero elements: 509
Sparsity: 50.29%

Sparse A (top-left 4x4):
[[ -8 -73   0   0]
 [-78 -44 105   0]
 [104 -63   0   0]
 [-19  77   0 -31]]

Sparse A (top-left 4x4, Hex):
F8 B7 00 00
B2 D4 69 00
68 C1 00 00
ED 4D 00 E1


In [26]:

def matrix_to_csr(matrix):
    """
    행렬을 CSR (Compressed Sparse Row) 포맷으로 변환
    
    Parameters:
        matrix: 입력 행렬 (2D numpy array)
    
    Returns:
        values: 0이 아닌 값들의 배열
        col_indices: 각 값의 열 인덱스
        row_ptr: 각 행의 시작 위치 (길이 = 행 수 + 1)
    """
    rows, cols = matrix.shape
    values = []
    col_indices = []
    row_ptr = [0]
    
    for i in range(rows):
        for j in range(cols):
            if matrix[i, j] != 0:
                values.append(int(matrix[i, j]))
                col_indices.append(j)
        row_ptr.append(len(values))
    
    return np.array(values), np.array(col_indices), np.array(row_ptr)

# sparse_a를 CSR 포맷으로 변환
csr_values, csr_col_indices, csr_row_ptr = matrix_to_csr(sparse_a)

print(f"=== Sparse A CSR Format ===")
print(f"Original shape: {sparse_a.shape}")
print(f"Total elements: {sparse_a.size}")
print(f"Non-zero elements: {len(csr_values)}")
print(f"Sparsity: {100 * (1 - len(csr_values) / sparse_a.size):.2f}%")

print(f"\nValues (first 20): {csr_values[:20]}")
print(f"Col indices (first 20): {csr_col_indices[:20]}")
print(f"Row pointers (first 10): {csr_row_ptr[:10]}")

# 16진수로 출력
print(f"\nValues (Hex, first 20):")
print(" ".join(f"{v & 0xFF:02X}" for v in csr_values[:20]))

print(f"\nCol indices (Hex, first 20):")
print(" ".join(f"{c:02X}" for c in csr_col_indices[:20]))

print(f"\nRow pointers (Hex, first 10):")
print(" ".join(f"{r:04X}" for r in csr_row_ptr[:10]))

# 검증: CSR에서 원본 행렬 복원
def csr_to_matrix(values, col_indices, row_ptr, shape):
    """CSR 포맷에서 원본 행렬 복원"""
    matrix = np.zeros(shape, dtype=np.int8)
    for i in range(len(row_ptr) - 1):
        for j in range(row_ptr[i], row_ptr[i + 1]):
            matrix[i, col_indices[j]] = values[j]
    return matrix

# 복원 검증 - sparse_a와 비교해야 함!
restored_a = csr_to_matrix(csr_values, csr_col_indices, csr_row_ptr, sparse_a.shape)
print(f"\nReconstruction check: {np.array_equal(sparse_a, restored_a)}")

=== Sparse A CSR Format ===
Original shape: (32, 32)
Total elements: 1024
Non-zero elements: 509
Sparsity: 50.29%

Values (first 20): [  -8  -73  -39   -6  -40  -85   16  -48  111   18   59 -128 -124  -78
  -44  105   27    8   20 -115]
Col indices (first 20): [ 0  1  4  6  7 11 17 19 20 23 24 29 31  0  1  2  4  6  8 10]
Row pointers (first 10): [  0  13  35  51  66  79  93 110 130 145]

Values (Hex, first 20):
F8 B7 D9 FA D8 AB 10 D0 6F 12 3B 80 84 B2 D4 69 1B 08 14 8D

Col indices (Hex, first 20):
00 01 04 06 07 0B 11 13 14 17 18 1D 1F 00 01 02 04 06 08 0A

Row pointers (Hex, first 10):
0000 000D 0023 0033 0042 004F 005D 006E 0082 0091

Reconstruction check: True


In [27]:

def save_csr_to_coe(csr_values, csr_col_indices, csr_row_ptr, prefix="csr"):
    """
    CSR 포맷의 각 배열을 COE 파일로 저장
    
    Parameters:
        csr_values: 0이 아닌 값들 (8비트 부호 있는 정수)
        csr_col_indices: 열 인덱스 (5비트, 0~31)
        csr_row_ptr: 행 포인터 (10비트, 0~1023)
        prefix: 파일명 접두어
    """
    
    # 1. Values COE (8비트)
    with open(f"{prefix}_values.coe", 'w') as f:
        f.write("memory_initialization_radix=16;\n")
        f.write("memory_initialization_vector=\n")
        for i, val in enumerate(csr_values):
            hex_val = val & 0xFF  # 8비트 2의 보수
            if i < len(csr_values) - 1:
                f.write(f"{hex_val:02X},\n")
            else:
                f.write(f"{hex_val:02X};\n")
    print(f"{prefix}_values.coe 생성 완료! ({len(csr_values)}줄)")
    
    # 2. Column Indices COE (8비트로 저장, 실제 5비트 사용)
    with open(f"{prefix}_col_indices.coe", 'w') as f:
        f.write("memory_initialization_radix=16;\n")
        f.write("memory_initialization_vector=\n")
        for i, col in enumerate(csr_col_indices):
            if i < len(csr_col_indices) - 1:
                f.write(f"{col:02X},\n")
            else:
                f.write(f"{col:02X};\n")
    print(f"{prefix}_col_indices.coe 생성 완료! ({len(csr_col_indices)}줄)")
    
    # 3. Row Pointers COE (16비트로 저장)
    with open(f"{prefix}_row_ptr.coe", 'w') as f:
        f.write("memory_initialization_radix=16;\n")
        f.write("memory_initialization_vector=\n")
        for i, ptr in enumerate(csr_row_ptr):
            if i < len(csr_row_ptr) - 1:
                f.write(f"{ptr:04X},\n")
            else:
                f.write(f"{ptr:04X};\n")
    print(f"{prefix}_row_ptr.coe 생성 완료! ({len(csr_row_ptr)}줄)")

# CSR 포맷을 COE 파일로 저장
save_csr_to_coe(csr_values, csr_col_indices, csr_row_ptr, "sparse_a_csr")

# 검증 출력
print("\n=== CSR COE 파일 검증 ===")
print(f"Values (first 10): {[f'{v & 0xFF:02X}' for v in csr_values[:10]]}")
print(f"Col indices (first 10): {[f'{c:02X}' for c in csr_col_indices[:10]]}")
print(f"Row pointers (first 5): {[f'{r:04X}' for r in csr_row_ptr[:5]]}")

sparse_a_csr_values.coe 생성 완료! (509줄)
sparse_a_csr_col_indices.coe 생성 완료! (509줄)
sparse_a_csr_row_ptr.coe 생성 완료! (33줄)

=== CSR COE 파일 검증 ===
Values (first 10): ['F8', 'B7', 'D9', 'FA', 'D8', 'AB', '10', 'D0', '6F', '12']
Col indices (first 10): ['00', '01', '04', '06', '07', '0B', '11', '13', '14', '17']
Row pointers (first 5): ['0000', '000D', '0023', '0033', '0042']


In [28]:
print(f"\nSparse A (top-left 4x4):")
print(restored_a[:4, :4])
restored_a_hex = restored_a.astype(h_type)

print(f"\nSparse A (top-left 4x4, Hex):")
for row in restored_a_hex[:4, :4]:
    print(" ".join(f"{val:02X}" for val in row))


Sparse A (top-left 4x4):
[[ -8 -73   0   0]
 [-78 -44 105   0]
 [104 -63   0   0]
 [-19  77   0 -31]]

Sparse A (top-left 4x4, Hex):
F8 B7 00 00
B2 D4 69 00
68 C1 00 00
ED 4D 00 E1


In [29]:
# 행렬 A와 B의 좌측 상위 4x4 출력
print("Matrix A (top-left 4x4):")
print(matrix_a[:4, :4])

print("\nMatrix B (top-left 4x4):")
print(matrix_b[:4, :4])

# 16진수로도 출력
print("\nMatrix A (top-left 4x4, Hex):")
for row in matrix_a_hex[:4, :4]:
    print(" ".join(f"{val:02X}" for val in row))

print("\nMatrix B (top-left 4x4, Hex):")
for row in matrix_b_hex[:4, :4]:
    print(" ".join(f"{val:02X}" for val in row))

Matrix A (top-left 4x4):
[[  -8  -73  112    4]
 [ -78  -44  105  -24]
 [ 104  -63 -108   25]
 [ -19   77   13  -31]]

Matrix B (top-left 4x4):
[[  50  -73   70   18]
 [ 110   45 -108  -37]
 [ -76  -65   81  -89]
 [  -4   -1  -15  108]]

Matrix A (top-left 4x4, Hex):
F8 B7 70 04
B2 D4 69 E8
68 C1 94 19
ED 4D 0D E1

Matrix B (top-left 4x4, Hex):
32 B7 46 12
6E 2D 94 DB
B4 BF 51 A7
FC FF F1 6C


In [30]:

def save_matrix_to_coe(matrix, filename, input_width=8):
    """
    행렬을 COE 파일로 저장 (2개 원소를 16비트로 결합)
    첫번째 원소 -> 하위 8비트, 두번째 원소 -> 상위 8비트
    
    Parameters:
        matrix: 저장할 행렬 (int8)
        filename: 출력 파일명
        input_width: 원소당 비트 수 (기본 8비트)
    """
    # uint8로 변환 (2의 보수 유지)
    matrix_hex = matrix.astype(np.uint8)
    rows, cols = matrix.shape
    
    with open(filename, 'w') as f:
        f.write("memory_initialization_radix=16;\n")
        f.write("memory_initialization_vector=\n")
        total_lines = (rows * cols) // 2
        line_idx = 0
        
        for i in range(rows):
            for j in range(0, cols, 2):
                low_byte = int(matrix_hex[i, j])      # 첫번째 원소 -> 하위 8비트
                high_byte = int(matrix_hex[i, j+1])   # 두번째 원소 -> 상위 8비트
                combined = (high_byte << input_width) | low_byte
                line_idx += 1
                if line_idx < total_lines:
                    f.write(f"{combined:04X},\n")
                else:
                    f.write(f"{combined:04X};\n")
    
    print(f"{filename} 생성 완료! ({total_lines}줄)")

# matrix_a와 matrix_b_trans를 COE 파일로 저장
save_matrix_to_coe(matrix_a, "matrix_a.coe", input_width)
save_matrix_to_coe(matrix_b_trans, "matrix_b_trans.coe", input_width)

# 검증 출력
print("\nMatrix A COE (first 4 lines):")
matrix_a_hex = matrix_a.astype(np.uint8)
for i in range(4):
    low = int(matrix_a_hex[0, i*2])
    high = int(matrix_a_hex[0, i*2+1])
    print(f"  A[0,{i*2}]={low:02X}, A[0,{i*2+1}]={high:02X} -> {(high<<input_width)|low:04X}")

print("\nMatrix B_trans COE (first 4 lines):")
matrix_b_trans_hex = matrix_b_trans.astype(np.uint8)
for i in range(4):
    low = int(matrix_b_trans_hex[0, i*2])
    high = int(matrix_b_trans_hex[0, i*2+1])
    print(f"  B_trans[0,{i*2}]={low:02X}, B_trans[0,{i*2+1}]={high:02X} -> {(high<<input_width)|low:04X}")

matrix_a.coe 생성 완료! (512줄)
matrix_b_trans.coe 생성 완료! (512줄)

Matrix A COE (first 4 lines):
  A[0,0]=F8, A[0,1]=B7 -> B7F8
  A[0,2]=70, A[0,3]=04 -> 0470
  A[0,4]=D9, A[0,5]=DB -> DBD9
  A[0,6]=FA, A[0,7]=D8 -> D8FA

Matrix B_trans COE (first 4 lines):
  B_trans[0,0]=32, B_trans[0,1]=6E -> 6E32
  B_trans[0,2]=B4, B_trans[0,3]=FC -> FCB4
  B_trans[0,4]=47, B_trans[0,5]=61 -> 6147
  B_trans[0,6]=E3, B_trans[0,7]=A4 -> A4E3


In [31]:
# C 행렬 계산 (A * B 행렬 곱)
# 8비트 부호 있는 정수 곱셈 결과는 최대 32비트 필요
if (result_width==32):
    d_type_r=np.int32
    # h_type_r=np.uint8 # uint8로 수정

# int16으로 변환 후 행렬 곱 (오버플로우 방지)
matrix_c = np.matmul(matrix_a.astype(d_type_r), matrix_b.astype(d_type_r))

# 32비트 2의 보수로 변환하는 함수
def to_unsigned_hex(val):
    val = int(val)
    if val < 0:
        return val & 0xFFFFFFFF
    return val

print("Matrix C (first 4x4):")
for i in range(4):
    print(" ".join(f"{to_unsigned_hex(matrix_c[i,j]):08X}" for j in range(4)))

print(f"\nMatrix C shape: {matrix_c.shape}")
print(f"Matrix C min: {matrix_c.min()}, max: {matrix_c.max()}")

# matrix_c.coe 생성 (32비트, 행 우선, 한 줄에 한 원소)
with open("matrix_c.coe", 'w') as f:
    f.write("memory_initialization_radix=16;\n")
    f.write("memory_initialization_vector=\n")
    total_lines = mat_size * mat_size  # 1024줄
    line_idx = 0
    for i in range(mat_size):
        for j in range(mat_size):
            # 32비트 2의 보수 형식으로 변환
            val = to_unsigned_hex(matrix_c[i, j])
            line_idx += 1
            if line_idx < total_lines:
                f.write(f"{val:08X},\n")
            else:
                f.write(f"{val:08X};\n")

print(f"\nmatrix_c.coe 파일 생성 완료! ({total_lines}줄)")

# 검증 출력
print("\nMatrix C COE (first 4 elements):")
for i in range(4):
    val = int(matrix_c[0, i])
    hex_val = to_unsigned_hex(val)
    print(f"  C[0,{i}] = {val} -> {hex_val:08X}")

Matrix C (first 4x4):
000000CA 0000176B FFFFED6B FFFF3D0A
FFFF884D 0000B3F3 00007924 FFFFB417
FFFFAB88 00006D47 0001201C 0000785E
00007FEB 00004C22 FFFFB5A8 00000086

Matrix C shape: (32, 32)
Matrix C min: -106538, max: 101461

matrix_c.coe 파일 생성 완료! (1024줄)

Matrix C COE (first 4 elements):
  C[0,0] = 202 -> 000000CA
  C[0,1] = 5995 -> 0000176B
  C[0,2] = -4757 -> FFFFED6B
  C[0,3] = -49910 -> FFFF3D0A


In [32]:
# sparse_a와 matrix_b의 행렬 곱 계산
# 오버플로우 방지를 위해 int32로 변환
sparse_c = np.matmul(sparse_a.astype(d_type_r), matrix_b.astype(d_type_r))

print("=== Sparse A × Matrix B ===")
print(f"Sparse C shape: {sparse_c.shape}")
print(f"Sparse C min: {sparse_c.min()}, max: {sparse_c.max()}")

# 32비트 2의 보수로 변환하는 함수
def to_unsigned_hex(val):
    val = int(val)
    if val < 0:
        return val & 0xFFFFFFFF
    return val

print("\nSparse C (first 4x4, Decimal):")
print(sparse_c[:4, :4])

print("\nSparse C (first 4x4, Hex):")
for i in range(4):
    print(" ".join(f"{to_unsigned_hex(sparse_c[i,j]):08X}" for j in range(4)))

# sparse_c.coe 생성
with open("sparse_c.coe", 'w') as f:
    f.write("memory_initialization_radix=16;\n")
    f.write("memory_initialization_vector=\n")
    total_lines = mat_size * mat_size
    line_idx = 0
    for i in range(mat_size):
        for j in range(mat_size):
            val = to_unsigned_hex(sparse_c[i, j])
            line_idx += 1
            if line_idx < total_lines:
                f.write(f"{val:08X},\n")
            else:
                f.write(f"{val:08X};\n")

print(f"\nsparse_c.coe 파일 생성 완료! ({total_lines}줄)")

=== Sparse A × Matrix B ===
Sparse C shape: (32, 32)
Sparse C min: -94225, max: 78780

Sparse C (first 4x4, Decimal):
[[  5424  26699  11929  -3811]
 [ -3509  32357  15640 -35274]
 [-22072   1993  47835  19290]
 [-22064   5713  -7309  18948]]

Sparse C (first 4x4, Hex):
00001530 0000684B 00002E99 FFFFF11D
FFFFF24B 00007E65 00003D18 FFFF7636
FFFFA9C8 000007C9 0000BADB 00004B5A
FFFFA9D0 00001651 FFFFE373 00004A04

sparse_c.coe 파일 생성 완료! (1024줄)


In [33]:

def csr_matmul(csr_values, csr_col_indices, csr_row_ptr, matrix_b):
    """
    CSR 포맷의 희소 행렬 A와 밀집 행렬 B의 곱셈
    
    Parameters:
        csr_values: A의 0이 아닌 값들
        csr_col_indices: 각 값의 열 인덱스
        csr_row_ptr: 각 행의 시작 위치
        matrix_b: 밀집 행렬 B (N x M)
    
    Returns:
        result: A × B 결과 행렬
    """
    num_rows = len(csr_row_ptr) - 1
    num_cols = matrix_b.shape[1]
    result = np.zeros((num_rows, num_cols), dtype=np.int32)
    
    for i in range(num_rows):
        # i번째 행의 non-zero 원소들
        row_start = csr_row_ptr[i]
        row_end = csr_row_ptr[i + 1]
        
        for idx in range(row_start, row_end):
            a_val = csr_values[idx]         # A[i, k]의 값
            k = csr_col_indices[idx]        # A[i, k]의 열 인덱스 k
            
            # C[i, :] += A[i, k] * B[k, :]
            result[i, :] += int(a_val) * matrix_b[k, :].astype(np.int32)
    
    return result

# CSR 형태로 행렬 곱 계산
sparse_c_csr = csr_matmul(csr_values, csr_col_indices, csr_row_ptr, matrix_b)

print("=== CSR-based Matrix Multiplication ===")
print(f"Result shape: {sparse_c_csr.shape}")
print(f"Result min: {sparse_c_csr.min()}, max: {sparse_c_csr.max()}")

print("\nSparse C via CSR (first 4x4, Decimal):")
print(sparse_c_csr[:4, :4])

print("\nSparse C via CSR (first 4x4, Hex):")
for i in range(4):
    print(" ".join(f"{to_unsigned_hex(sparse_c_csr[i,j]):08X}" for j in range(4)))

# 검증: numpy matmul 결과와 비교
print(f"\nVerification (CSR vs numpy): {np.array_equal(sparse_c, sparse_c_csr)}")

# 연산 횟수 비교
dense_ops = mat_size * mat_size * mat_size  # 일반 행렬곱
sparse_ops = len(csr_values) * mat_size     # CSR 기반 곱셈
print(f"\nDense multiplication ops: {dense_ops}")
print(f"CSR-based multiplication ops: {sparse_ops}")
print(f"Speedup ratio: {dense_ops / sparse_ops:.2f}x")

=== CSR-based Matrix Multiplication ===
Result shape: (32, 32)
Result min: -94225, max: 78780

Sparse C via CSR (first 4x4, Decimal):
[[  5424  26699  11929  -3811]
 [ -3509  32357  15640 -35274]
 [-22072   1993  47835  19290]
 [-22064   5713  -7309  18948]]

Sparse C via CSR (first 4x4, Hex):
00001530 0000684B 00002E99 FFFFF11D
FFFFF24B 00007E65 00003D18 FFFF7636
FFFFA9C8 000007C9 0000BADB 00004B5A
FFFFA9D0 00001651 FFFFE373 00004A04

Verification (CSR vs numpy): True

Dense multiplication ops: 32768
CSR-based multiplication ops: 16288
Speedup ratio: 2.01x


In [34]:
def coe_to_hex(coe_filename, hex_filename):
    """
    COE 파일을 $readmemh용 HEX 파일로 변환
    COE 헤더를 제거하고 순수 hex 데이터만 저장
    
    Parameters:
        coe_filename: 입력 COE 파일 경로
        hex_filename: 출력 HEX 파일 경로
    """
    with open(coe_filename, 'r') as f_in:
        lines = f_in.readlines()
    
    hex_data = []
    data_started = False
    
    for line in lines:
        line = line.strip()
        
        # 빈 줄 무시
        if not line:
            continue
        
        # 헤더 라인 건너뛰기
        if 'memory_initialization' in line.lower():
            if 'vector' in line.lower():
                data_started = True
            continue
        
        # 데이터 시작 후
        if data_started:
            # 쉼표, 세미콜론 제거
            clean_line = line.replace(',', '').replace(';', '').strip()
            if clean_line:
                hex_data.append(clean_line)
    
    # HEX 파일로 저장
    with open(hex_filename, 'w') as f_out:
        for data in hex_data:
            f_out.write(data + '\n')
    
    print(f"{coe_filename} -> {hex_filename} 변환 완료! ({len(hex_data)}줄)")
    return len(hex_data)

# 모든 COE 파일을 HEX 파일로 변환
coe_files = [
    ("sparse_c.coe", "sparse_c.hex"),
    # ("sparse_a_csr_values.coe", "sparse_a_csr_values.hex"),
    # ("sparse_a_csr_col_indices.coe", "sparse_a_csr_col_indices.hex"),
    # ("sparse_a_csr_row_ptr.coe", "sparse_a_csr_row_ptr.hex"),
    # ("matrix_b_trans.coe", "matrix_b_trans.hex"),
]

print("=== COE to HEX 변환 ===")
for coe_file, hex_file in coe_files:
    try:
        coe_to_hex(coe_file, hex_file)
    except FileNotFoundError:
        print(f"{coe_file} 파일을 찾을 수 없습니다.")

# 변환 결과 확인 (첫 5줄)
print("\n=== sparse_c.hex 확인 (첫 5줄) ===")
try:
    with open("sparse_c.hex", 'r') as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            print(line.strip())
except FileNotFoundError:
    print("sparse_c.hex 파일을 찾을 수 없습니다.")

=== COE to HEX 변환 ===
sparse_c.coe -> sparse_c.hex 변환 완료! (1024줄)

=== sparse_c.hex 확인 (첫 5줄) ===
00001530
0000684B
00002E99
FFFFF11D
FFFFF402
